In [1]:
import cv2
import os
from pathlib import Path
from typing import List, Dict, Tuple
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [2]:
class BrailleCropper:
    def __init__(self, overlap_ratio: float = 0.1, crop_size: int = 224, min_annotation_overlap: float = 0.8):
        self.overlap_ratio = overlap_ratio
        self.crop_size = crop_size
        self.min_annotation_overlap = min_annotation_overlap

    def parse_annotations(self, annotation_path: str, img_width: int, img_height: int) -> List[Dict]:
        """Parse YOLO txt format and convert to absolute pixel coords."""
        if not os.path.exists(annotation_path):
            return []
        annotations = []
        with open(annotation_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) != 5:
                    continue
                class_id, cx, cy, w, h = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                abs_cx = cx * img_width
                abs_cy = cy * img_height
                abs_w  = w  * img_width
                abs_h  = h  * img_height
                annotations.append({
                    'left':   abs_cx - abs_w / 2,
                    'top':    abs_cy - abs_h / 2,
                    'right':  abs_cx + abs_w / 2,
                    'bottom': abs_cy + abs_h / 2,
                    'class_id': class_id
                })
        return annotations

    def calculate_crop_positions(self, img_height: int, img_width: int) -> List[Tuple[int,int,int,int]]:
        step_y = max(int(self.crop_size * (1 - self.overlap_ratio)), 1)
        step_x = max(int(self.crop_size * (1 - self.overlap_ratio)), 1)
        y_positions = list(range(0, img_height - self.crop_size + 1, step_y))
        x_positions = list(range(0, img_width - self.crop_size + 1, step_x))
        if y_positions and y_positions[-1] + self.crop_size < img_height:
            y_positions.append(img_height - self.crop_size)
        if x_positions and x_positions[-1] + self.crop_size < img_width:
            x_positions.append(img_width - self.crop_size)
        if not y_positions: y_positions = [0]
        if not x_positions: x_positions = [0]
        crops = [(y, min(y + self.crop_size, img_height),
                  x, min(x + self.crop_size, img_width)) for y in y_positions for x in x_positions]
        return crops

    def bbox_overlap_ratio(self, annotation: Dict, crop_coords: Tuple[int,int,int,int]) -> float:
        y1, y2, x1, x2 = crop_coords
        ann_x1, ann_y1 = annotation['left'],  annotation['top']
        ann_x2, ann_y2 = annotation['right'], annotation['bottom']
        inter_x1, inter_y1 = max(ann_x1, x1), max(ann_y1, y1)
        inter_x2, inter_y2 = min(ann_x2, x2), min(ann_y2, y2)
        if inter_x2 <= inter_x1 or inter_y2 <= inter_y1:
            return 0.0
        intersection_area = (inter_x2 - inter_x1) * (inter_y2 - inter_y1)
        annotation_area   = (ann_x2 - ann_x1)     * (ann_y2 - ann_y1)
        return 0.0 if annotation_area <= 0 else intersection_area / annotation_area

    def transform_annotations_to_crop(self, annotations: List[Dict],
                                       crop_coords: Tuple[int,int,int,int]) -> List[Dict]:
        """Filter annotations by overlap and return them in YOLO normalized coords relative to crop."""
        y1, y2, x1, x2 = crop_coords
        crop_h, crop_w = y2 - y1, x2 - x1
        if crop_h <= 0 or crop_w <= 0:
            return []
        transformed = []
        for ann in annotations:
            if self.bbox_overlap_ratio(ann, crop_coords) >= self.min_annotation_overlap:
                # Clip to crop boundary and shift origin
                crop_x1 = max(ann['left'],   x1) - x1
                crop_y1 = max(ann['top'],    y1) - y1
                crop_x2 = min(ann['right'],  x2) - x1
                crop_y2 = min(ann['bottom'], y2) - y1
                if crop_x2 > crop_x1 and crop_y2 > crop_y1:
                    # Convert back to YOLO normalized format
                    center_x = (crop_x1 + crop_x2) / 2 / crop_w
                    center_y = (crop_y1 + crop_y2) / 2 / crop_h
                    width    = (crop_x2 - crop_x1) / crop_w
                    height   = (crop_y2 - crop_y1) / crop_h
                    transformed.append({
                        'class_id': ann['class_id'],
                        'center_x': center_x,
                        'center_y': center_y,
                        'width':    width,
                        'height':   height,
                    })
        return transformed

    def crop_image_with_annotations(
        self,
        image_path: str,
        annotation_path: str,
        output_dir: str,
        annotation_dir: str,
        prefix: str = None
    ) -> dict:
        img = cv2.imread(image_path)
        if img is None:
            raise FileNotFoundError(f"Image not found: {image_path}")

        img_height, img_width = img.shape[:2]

        annotations    = self.parse_annotations(annotation_path, img_width, img_height)
        crop_positions = self.calculate_crop_positions(img_height, img_width)

        os.makedirs(output_dir,    exist_ok=True)
        os.makedirs(annotation_dir, exist_ok=True)

        if prefix is None:
            prefix = Path(image_path).parent.name[:2]
        base_name = f"{prefix}.{Path(image_path).stem}" if prefix else Path(image_path).stem

        results = []
        for i, (y1, y2, x1, x2) in enumerate(crop_positions):
            cropped_img = img[y1:y2, x1:x2]
            if cropped_img.shape[:2] != (self.crop_size, self.crop_size):
                cropped_img = cv2.resize(cropped_img, (self.crop_size, self.crop_size))

            crop_annotations = self.transform_annotations_to_crop(annotations, (y1, y2, x1, x2))

            crop_filename      = f"{base_name}_{i:02d}.jpg"
            annotation_filename = f"{base_name}_{i:02d}.txt"

            cv2.imwrite(os.path.join(output_dir, crop_filename), cropped_img)

            with open(os.path.join(annotation_dir, annotation_filename), 'w') as f:
                for ann in crop_annotations:
                    f.write(f"{ann['class_id']} {ann['center_x']:.6f} {ann['center_y']:.6f} "
                            f"{ann['width']:.6f} {ann['height']:.6f}\n")

            results.append({
                'crop_id':        i,
                'image_path':     os.path.join(output_dir, crop_filename),
                'annotation_path': os.path.join(annotation_dir, annotation_filename),
                'crop_coords':    (y1, y2, x1, x2),
                'num_annotations': len(crop_annotations)
            })

        return {
            'original_image': image_path,
            'crop_size':      (self.crop_size, self.crop_size),
            'total_crops':    len(results),
            'crops':          results
        }

In [3]:
CROP_SIZE = 640
OVERLAP   = 0.1
MIN_ANNOTATION_OVERLAP = 0.8

cropper = BrailleCropper(
    crop_size=CROP_SIZE,
    overlap_ratio=OVERLAP,
    min_annotation_overlap=MIN_ANNOTATION_OVERLAP
)

def process_dataset_local(images_dir, ann_dir, output_images_dir, output_ann_dir,
                           process_missing_annotations=True):
    all_files = []
    for root_dir, _, files in os.walk(images_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_files.append((root_dir, file))

    print(f"Total images found: {len(all_files)}")
    skipped_missing   = 0
    processed_images  = 0
    total_crops_saved = 0

    for root_dir, file in tqdm(all_files, desc="Cropping locally"):
        img_path = os.path.join(root_dir, file)
        rel_path = os.path.relpath(root_dir, images_dir)

        ann_path = os.path.join(ann_dir, rel_path, Path(file).with_suffix('.txt').name)

        if not os.path.exists(ann_path):
            print(f"[!] Missing annotation for {img_path} -> {ann_path}")
            if not process_missing_annotations:
                skipped_missing += 1
                continue

        prefix = rel_path[:2] if rel_path != '.' else ''
        res = cropper.crop_image_with_annotations(
            image_path=img_path,
            annotation_path=ann_path,
            output_dir=str(output_images_dir),
            annotation_dir=str(output_ann_dir),
            prefix=prefix
        )
        total_crops_saved += res.get('total_crops', 0)
        processed_images  += 1

    print("\nSUMMARY")
    print(f"Images processed         : {processed_images}")
    print(f"Images skipped (missing) : {skipped_missing}")
    print(f"Total crops created      : {total_crops_saved}")
    print(f"Cropped dataset ready at : {output_images_dir}")

In [4]:
PROJECT_ROOT = Path("../data")
DATASET_DIR = PROJECT_ROOT / "synthetic1"
IMAGES_DIR = DATASET_DIR / "images/train"
ANNOTATIONS_DIR = DATASET_DIR / "labels/train"

OUTPUT_DIR = PROJECT_ROOT / "cropped"
OUTPUT_IMAGES_DIR = OUTPUT_DIR / "images"
OUTPUT_LABELS_DIR = OUTPUT_DIR / "labels"
os.makedirs(OUTPUT_IMAGES_DIR, exist_ok=True)
os.makedirs(OUTPUT_LABELS_DIR, exist_ok=True)

process_dataset_local(IMAGES_DIR, ANNOTATIONS_DIR, OUTPUT_IMAGES_DIR, OUTPUT_LABELS_DIR)

Total images found: 1000


Cropping locally:   0%|          | 0/1000 [00:00<?, ?it/s]

Cropping locally: 100%|██████████| 1000/1000 [00:19<00:00, 52.52it/s]


SUMMARY
Images processed         : 1000
Images skipped (missing) : 0
Total crops created      : 6000
Cropped dataset ready at : ../data/cropped/images


In [5]:
import os, random

all_crops = [
    f for _, _, files in os.walk(IMAGES_DIR)
    for f in files if f.lower().endswith('.jpg')
]

print(f"Total cropped images: {len(all_crops)}")

for name in random.sample(all_crops, min(10, len(all_crops))):
    print(name)

Total cropped images: 1000
sample_00660.jpg
sample_00198.jpg
sample_00405.jpg
sample_00785.jpg
sample_00509.jpg
sample_00611.jpg
sample_00199.jpg
sample_00443.jpg
sample_00346.jpg
sample_00245.jpg


In [6]:
from pathlib import Path
from sklearn.model_selection import train_test_split
import shutil
from tqdm import tqdm

IMAGES_DIR = OUTPUT_DIR / "images"
LABELS_DIR = OUTPUT_DIR / "labels"

train_split, val_split, test_split = 0.7, 0.2, 0.1
random_seed = 42

for txt_file in IMAGES_DIR.rglob("*.txt"):
    txt_file.unlink()

all_images = sorted([p for p in IMAGES_DIR.rglob("*.jpg")])

train_imgs, temp_imgs = train_test_split(
    all_images,
    test_size=(1 - train_split),
    random_state=random_seed,
    shuffle=True
)

val_imgs, test_imgs = train_test_split(
    temp_imgs,
    test_size=test_split / (val_split + test_split),
    random_state=random_seed,
    shuffle=True
)

splits = {
    "train": train_imgs,
    "val": val_imgs,
    "test": test_imgs
}

for split_name, img_list in splits.items():
    img_out_dir = OUTPUT_DIR / "images" / split_name
    label_out_dir = OUTPUT_DIR / "labels" / split_name

    img_out_dir.mkdir(parents=True, exist_ok=True)
    label_out_dir.mkdir(parents=True, exist_ok=True)

    for img_path in tqdm(img_list, desc=split_name):
        dst_img = img_out_dir / img_path.name
        shutil.move(str(img_path), str(dst_img))

        label_path = LABELS_DIR / f"{img_path.stem}.txt"

        if label_path.exists():
            dst_label = label_out_dir / label_path.name
            shutil.move(str(label_path), str(dst_label))

test: 100%|██████████| 601/601 [00:00<00:00, 16164.42it/s]


In [7]:
# from ultralytics import YOLO

# model = YOLO(r"runs\detect\yolov11s_5123\weights\best.pt")

# metrics = model.val(
#     data="dataset.yaml",
#     split="test",
#     imgsz=512,
#     batch=2,
#     device="0"
# )

# results = metrics.mean_results()
# if len(results) == 5:
#     precision, recall, map50, map50_95, f1 = results
# else:
#     precision, recall, map50, map50_95 = results
#     f1 = 2 * (precision * recall) / (precision + recall + 1e-16)  # compute F1 manually

# print(f"Precision: {precision:.3f}")
# print(f"Recall:    {recall:.3f}")
# print(f"mAP50:     {map50:.3f}")
# print(f"mAP50-95:  {map50_95:.3f}")
# print(f"F1 Score:  {f1:.3f}")